In [1]:
!pip install pandas numpy datasets pyarrow 

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
import hashlib
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 180)

DATASET_NAME = 'chillies/IELTS-writing-task-2-evaluation'
ds = load_dataset(DATASET_NAME)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'essay', 'evaluation', 'band'],
        num_rows: 9833
    })
    test: Dataset({
        features: ['prompt', 'essay', 'evaluation', 'band'],
        num_rows: 491
    })
})


In [4]:
type(ds)

datasets.dataset_dict.DatasetDict

In [5]:
split_name = list(ds.keys())[0]
df = ds[split_name].to_pandas()

In [7]:
df

,prompt,essay,evaluation,band
0,"Interviews form the basic criteria for most large companies. However, some people think that the interview is not a reliable method of choosing whom to employ and there are oth...","It is believed by some experts that the traditional approach of recruiting candidates which is interviewing is the best way, whereas others think different methods such as exam...",**Task Achievement: [7]**\nThe essay effectively addresses the given task. The candidate clearly states their position in the introduction and provides relevant arguments and e...,7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r
1,"Interviews form the basic selecting criteria for most large companies. However, some people think that the interview is not a reliable method of choosing whom to employ and the...","Nowadays numerous huge firms allocate an interview form as the basic choosing criteria. Whereas, a group of the public believe that this form is no longer up-to-the-minute and ...",**Task Achievement:** 5.0\n- The candidate has effectively addressed the given task by providing a clear and relevant response that covers all aspects of the task.\n- The essay...,5.0\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r
2,"Interview form the basic selection criteria for most large companies. However, some people think that interview is not a reliable method of choosing whom to employ and there ar...",The interview section is the most vital part of the hiring process. It is the time for knowing the interviewee and also for the person being interviewed to learn some informati...,## Task Achievement:\n- The candidate has effectively addressed the given task by presenting a clear stance on the reliability of interviews as a method of employee selection a...,5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r
3,"Interviews form the basic selection criteria for most large companies. However, some poeple think that interviews is not a reliable mthod of choosing whom to employ and there a...","It is argued that the best method to recruit employees for the job is arranging interviewing sessions which are usually preferred by most large-scale companies, whereas the res...",## Task Achievement:\n- The candidate has adequately addressed the task by presenting arguments for and against the reliability of interviews as a method of employee selection....,5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r
4,Interviews from the basic selecting criteria for most large companies. However some people think that interview not a reliable method of chosing whom to employ and there are ot...,Nowadays many companies conduct interviews before hiring their employees. it is also said that interview is not a reliable method. \n\nBecause it is related to the company's em...,"**Task Achievement:**\n\nThe essay adequately addresses the task by discussing both sides of the argument regarding the reliability of interviews as a hiring method. However, t...",4\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r
...,...,...,...,...
9828,"Nations should spend more money on skills and vocational training for practical work, rather than on university education.\r\nTo what extent do you agree or disagree?","These days, many countries use a huge amount of money on providing lessons related to pragmatic skills, such as video editing class and coding class, and the government insist ...",### Task Achievement:\nThe essay addresses the task by discussing the benefits of spending more money on skills and vocational training for practical work rather than on univer...,8
9829,"Nations should spend more money on skills and vocational training for practical work, rather than on university education.\r\nTo what extent do you agree or disagree?\r\nRead m...",Skills are required in order to achieve success in life. It is said by some individuals that all the countries should focus on additional courses and development of skills r...,## Task Achievement:\n- The candidate has not effectively addressed the given task.\n- The essay fails to provide a clear st

In [11]:
df.duplicated().sum()

0

In [12]:
band_re = re.compile(r"(\d(?:\.\d)?)")

def parse_band(x):
    if x is None:
        return None
    s = str(x).strip()
    m = band_re.search(s)
    if not m:
        return None
    try:
        return float(m.group(1))
    except ValueError:
        return None

df = df.copy()
df["band_value"] = df["band"].map(parse_band)

df_75 = df[df["band_value"].notna() & (df["band_value"] >= 7.5)].copy()

print("rows_before:", len(df))
print("rows_band>=7.5:", len(df_75))
print("band_counts:\n", df_75["band_value"].value_counts().sort_index())

rows_before: 9833
rows_band>=7.5: 2346
band_counts:
 band_value
7.5    1105
8.0     700
8.5     433
9.0     108
Name: count, dtype: int64


In [13]:
df_75.to_json("../data/processed/task2_band7p5plus.jsonl", orient="records", lines=True, force_ascii=False)
df_75.to_csv("../data/processed/task2_band7p5plus.csv", index=False)

In [14]:
import pandas as pd
import re
import hashlib

required_markers = [
    "Task Achievement",
    "Coherence and Cohesion",
    "Lexical Resource",
    "Overall Band Score",
    "Grammatical Range and Accuracy",
]

ev = df_75["evaluation"].astype(str)
mask = pd.Series(True, index=df_75.index)
for m in required_markers:
    mask &= ev.str.contains(m, case=False, na=False)

df_m = df_75[mask].copy()

print("rows_after_markers:", len(df_m))

# 2) Length filter + dedup
def norm_text(s: str) -> str:
    s = str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def wc(x) -> int:
    return len(str(x).split())

df_m["essay_words"] = df_m["essay"].map(wc)

MIN_WORDS = 80
MAX_WORDS = 600
df_m = df_m[(df_m["essay_words"] >= MIN_WORDS) & (df_m["essay_words"] <= MAX_WORDS)].copy()

df_m["prompt_hash"] = df_m["prompt"].map(norm_text).map(sha256_text)
df_m["essay_hash"] = df_m["essay"].map(norm_text).map(sha256_text)

before_dedup = len(df_m)
df_ready = df_m.drop_duplicates(subset=["prompt_hash", "essay_hash"], keep="first").copy()

print("rows_after_len_filter:", before_dedup)
print("rows_after_dedup (df_ready):", len(df_ready))

# sanity check: markers should all be present now
presence_after = {m: df_ready["evaluation"].astype(str).str.contains(m, case=False, na=False).mean() for m in required_markers}
print("presence_after:", presence_after)

df_ready[["band_value", "essay_words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T

rows_after_markers: 2345
rows_after_len_filter: 2345
rows_after_dedup (df_ready): 2134
presence_after: {'Task Achievement': 1.0, 'Coherence and Cohesion': 1.0, 'Lexical Resource': 1.0, 'Overall Band Score': 1.0, 'Grammatical Range and Accuracy': 1.0}


,count,mean,std,min,50%,90%,95%,99%,max
band_value,2134.0,7.899485,0.446998,7.5,8.0,8.5,8.5,9.00,9.0
essay_words,2134.0,305.781162,41.254137,202.0,298.0,360.0,383.0,434.34,538.0


In [15]:
df_ready.head(5)

,prompt,essay,evaluation,band,band_value,essay_words,prompt_hash,essay_hash
0,"Interviews form the basic criteria for most large companies. However, some people think that the interview is not a reliable method of choosing whom to employ and there are oth...","It is believed by some experts that the traditional approach of recruiting candidates which is interviewing is the best way, whereas others think different methods such as exam...",**Task Achievement: [7]**\nThe essay effectively addresses the given task. The candidate clearly states their position in the introduction and provides relevant arguments and e...,7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r,7.5,357,ae7cd02374c180820d61e493b4c2818b4a233138ad3a815134b7aa5c28564fdc,b74c853dd5c14d73e04ec372b0a817dbd74fc4c32f65d519b08dfe15cafedccc
12,"Interviews form the basic selecting criteria for most large companies. However, some people think that the interview is not a reliable method of choosing whom to employ and the...","Nowadays, most companies employ workers after interviewing them. Certain individuals think this is not a perfect way of choosing workers since there are better ways of doing it...",## Task Achievement:\n- The candidate has effectively addressed the given task by presenting a clear position and supporting arguments.\n- The ideas presented are relevant and ...,7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r,7.5,321,a9f881ea279d65222c49c73ee9a34abe0a1987b5301b9d4bc9157c184d82e964,c0dbc2a18c5c265dfa41c9a43ac70025fb16a20df280ae08ef6655c6db8d2fef
16,Interviews form the basic selection criteria for most large companies. However some people think that interview is not a reliable method of choosing whom to employ.\nTo what ex...,"Interviews are typically the primary selection criteria for large companies. There is some criticism of using interviews to recruit employees, with the argument that this metho...",### Task Achievement: 8\n- The essay adequately addresses the task and presents relevant ideas in response to the prompt.\n- While the candidate effectively covers the main poi...,8\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r,8.0,341,330303b2beca8a7ee5b1c719db291c07e0d4faacbefc33698fff47368f0e1e0c,1cbabc646bc39eebd34af089fcb2cc91466d62c008dd6f874bcbecb8aed22853
17,Interviews form the basic selection criteria for most large companies. However some people think that interview is not a reliable method of choosing whom to employ.\nTo what ex...,Job interviews are commonly undertaken by companies of all sizes seeking candidates to fill a role. They often consist of dialogues between a Human Resources representative of ...,## Task Achievement:\n- The candidate has effectively addressed the given task by presenting a clear stance on the topic.\n- The ideas presented are relevant to the prompt and ...,7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r,7.5,372,330303b2beca8a7ee5b1c719db291c07e0d4faacbefc33698fff47368f0e1e0c,f207171cb079ab5e046a5cc1f4b07e254bd954a33f6ed38aecf4370452875b4b
19,"interviews form the basic selecting criteria for most large campaniles.\nHowever, some people think the interview is not o reliable method of\nchoosing whom lo employ and there...","Nowadays, mega-corporations may adopt several procedures to recruit employees. One of the most common ways in well-known firms is to select their candidates by utilizing a face...",## Task Achievement:\n- The candidate has adequately addressed the task and provided a clear stance on the topic.\n- The essay presents relevant ideas and arguments in response...,7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r,7.5,355,0debdcf847619b650a55ffe8dcb4ef05817877c3e73e0ef9f38147692f5621a3,3001f4d8a2e1bc9ed1cc29136570db40aed5c4cd5cbd6771355890504d527c8b


## Split evaluation into feedback cards + chunk essay by paragraph

Tạo 3 loại record chuẩn bị cho vector DB RAG:
- **essay_chunk**: paragraph-level chunks từ essay
- **feedback_card**: strengths / improvements / suggestions từ evaluation

In [16]:
import json
from pathlib import Path

def stable_id(prefix: str, s: str) -> str:
    return f"{prefix}_{sha256_text(s)[:16]}"

def split_paragraphs(essay: str) -> list:
    s = str(essay)
    if "\n\n" in s:
        return [p.strip() for p in s.split("\n\n") if p.strip()]
    return [re.sub(r"\s+", " ", s).strip()] if s.strip() else []

section_re = re.compile(r"^\s*(\*\*|#+)\s*(.+?)\s*(\*\*\s*)?$", re.IGNORECASE)

def extract_feedback_sections(evaluation: str) -> dict:
    text = str(evaluation)
    lines = [ln.rstrip() for ln in text.splitlines()]
    current = "raw"
    buckets = {"raw": []}

    def norm_key(title):
        t = title.lower()
        if "strength" in t: return "strengths"
        if "areas for improvement" in t or "improvement" in t: return "improvements"
        if "suggestion" in t: return "suggestions"
        if "additional" in t or "comment" in t: return "comments"
        return "other"

    for ln in lines:
        m = section_re.match(ln)
        if m:
            title = m.group(2).strip()
            current = norm_key(title)
            buckets.setdefault(current, [])
            continue
        buckets.setdefault(current, []).append(ln)

    return {k: "\n".join([x for x in v if x.strip()]).strip() for k, v in buckets.items() if any(x.strip() for x in v)}

essays_out = []
chunks_out = []
cards_out = []

for idx, row in df_ready.iterrows():
    prompt = re.sub(r"\s+", " ", str(row["prompt"])).strip()
    essay = re.sub(r"\s+", " ", str(row["essay"])).strip().replace("\r\n", "\n")
    evaluation = str(row["evaluation"])
    band = float(row["band_value"])

    base_id = stable_id("t2", row["prompt_hash"] + row["essay_hash"])

    essays_out.append({
        "id": base_id,
        "source_type": "essay",
        "task": "writing_task2",
        "band": band,
        "prompt": prompt,
        "text": essay,
        "evaluation": evaluation,
        "source": "chillies/IELTS-writing-task-2-evaluation",
    })

    for i, p in enumerate(split_paragraphs(essay)):
        if not p.strip():
            continue
        chunks_out.append({
            "id": f"{base_id}_p{i}",
            "source_type": "essay_chunk",
            "parent_id": base_id,
            "task": "writing_task2",
            "band": band,
            "prompt": prompt,
            "text": p,
            "chunk_index": i,
        })

    sections = extract_feedback_sections(evaluation)
    for sec in ["strengths", "improvements", "suggestions"]:
        if sec not in sections or not sections[sec]:
            continue
        cards_out.append({
            "id": f"{base_id}_{sec}",
            "source_type": "feedback_card",
            "card_type": sec,
            "parent_id": base_id,
            "task": "writing_task2",
            "band": band,
            "prompt": prompt,
            "text": sections[sec],
        })

print("essays:", len(essays_out))
print("essay_chunks:", len(chunks_out))
print("feedback_cards:", len(cards_out))

essays: 2134
essay_chunks: 2134
feedback_cards: 2035


## Export JSONL (sẵn sàng cho ingestion vào vector DB)

In [17]:
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

def write_jsonl(path: Path, rows: list):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(out_dir / "task2_band7p5plus_essays.jsonl", essays_out)
write_jsonl(out_dir / "task2_band7p5plus_essay_chunks.jsonl", chunks_out)
write_jsonl(out_dir / "task2_band7p5plus_feedback_cards.jsonl", cards_out)

summary = {
    "rows_ready": len(df_ready),
    "essays": len(essays_out),
    "essay_chunks": len(chunks_out),
    "feedback_cards": len(cards_out),
    "outputs": [
        str(out_dir / "task2_band7p5plus_essays.jsonl"),
        str(out_dir / "task2_band7p5plus_essay_chunks.jsonl"),
        str(out_dir / "task2_band7p5plus_feedback_cards.jsonl"),
    ],
}
out_dir.joinpath("task2_band7p5plus_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(summary)

{'rows_ready': 2134, 'essays': 2134, 'essay_chunks': 2134, 'feedback_cards': 2035, 'outputs': ['../data/processed/task2_band7p5plus_essays.jsonl', '../data/processed/task2_band7p5plus_essay_chunks.jsonl', '../data/processed/task2_band7p5plus_feedback_cards.jsonl']}
